# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing entities strictly by their `@id` fields.

### Dataset Source
The dataset is described by a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json)

It contains structured tabulations of clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata and schema
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
dataset_metadata = dataset.metadata
print(f"{dataset_metadata.name}: {dataset_metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll list all `RecordSet` objects (tables), show their `@id`, and the `field`/`column` entities inside them. This will guide us to reference entities by their `@id` in subsequent steps.

In [ ]:
# Get all record sets from the dataset schema
record_sets = dataset.record_sets
print("Available RecordSets:")

for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name','N/A')}")
    print("  Fields/Columns:")
    for field in rs.get('field', []):
        print(f"    - Field @id: {field['@id']} | Name: {field.get('name','N/A')} | DataType: {field.get('dataType','N/A')}")
    for col in rs.get('column', []):
        print(f"    - Column @id: {col['@id']} | Name: {col.get('name','N/A')} | DataType: {col.get('dataType','N/A')}")
    print("")

## 3. Data Extraction
Load data from each record set (table) into a DataFrame for analysis.

We refer to each table and field strictly by their `@id` as discovered above.

_Note_: If there are three tables (per metadata: clinical features, hormone levels/adrenal imaging, genetic variants), we extract all. Replace `<record_set_ids>` below with actual discovered `@id`s.

In [ ]:
# Get the list of record set @id fields
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Extract records for each record set, referencing via @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nFields for RecordSet @id: {record_set_id}")
    print(df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps.
- Filtering records based on field values
- Normalizing numeric fields
- Grouping/categorizing by key attributes

For demonstration, we'll select a numeric field and group field by their `@id` from one of the tables.

In [ ]:
# Choose a record set and fields for EDA
# For example, suppose the first record set contains a numeric field 'age at diagnosis' and a group field 'infertility'
# Reference strictly by @id

# Example placeholder for record set and field ids
selected_record_set_id = record_set_ids[0]  # Replace with actual @id if known, e.g., 'cr:ClinicalTable1'
df = dataframes[selected_record_set_id]

# Find candidate numeric and group fields
numeric_field_id = None
group_field_id = None
for rs in dataset.record_sets:
    if rs['@id'] == selected_record_set_id:
        for field in rs.get('field', []) + rs.get('column', []):
            # Example heuristics for choosing numeric and group fields
            if ('age' in field.get('name','').lower() or 'height' in field.get('name','').lower()) and field.get('dataType','').endswith('Integer'):
                numeric_field_id = field['@id']
            if ('infertility' in field.get('name','').lower() or 'phenotype' in field.get('name','').lower()):
                group_field_id = field['@id']
        break

# If not found, fallback to the first numeric and group field
if numeric_field_id is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if group_field_id is None:
    for col in df.columns:
        if df[col].dtype == 'object' and (col != numeric_field_id):
            group_field_id = col
            break

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

# Demonstrate filtering, normalization, grouping
if numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields by referencing fields by their `@id`.

For example, plot the distribution of age at diagnosis (or another numeric feature) and visualize the relationship with infertility status.

In [ ]:
if numeric_field_id and group_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"Distribution of {numeric_field_id} grouped by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()
elif numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    df[numeric_field_id].hist()
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, process, and visualize a FAIR^2 clinical dataset using `mlcroissant`.

- Entities were referenced strictly via their `@id`.
- Three structured tables were loaded, with each table and field extracted using its Croissant `@id`.
- Common exploratory analysis and visualization were performed using the actual schema.

This approach streamlines robust, reproducible analysis of clinical and genomic datasets defined in Croissant format.